<a href="https://colab.research.google.com/github/cswcjt/Fastcampus-ML/blob/main/oil_cjt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Data load

In [ ]:
# ????? 4? ??
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_path = "/content/drive/MyDrive/fastcamp/datas/open/"
train = pd.read_csv(base_path + "train.csv", encoding='cp949')
test = pd.read_csv(base_path + "test.csv", encoding='cp949')
submission = pd.read_csv(base_path + "sample_submission.csv", encoding='cp949')

In [ ]:
print("Train shape : ", train.shape)
print("Test shape : ", test.shape)

In [ ]:
test.head()

### 2. EDA : ?? -> ???? ?? ??

- ????? ??? ?? ???? ???? ?????.

- class imbalance, target distribution, outlier, correlation? ?????.

1) Target Variable : "y" is the variable we need to predict. 
- 0 means oil condition is normal
- 1 means oil condition is abnormal

In [ ]:
train.Y_LABEL.value_counts()

In [ ]:
plt.figure(figsize=(8,6))
plt.figure(figsize = (14, 7))
sns.countplot(data = train, x = train.Y_LABEL)
plt.show()

2) Now let us have a look at the data type of all the variables present in the dataset.
- ?? : 

int64 -> 31

categorical columns -> 2

float64 -> 21 

In [ ]:
dtype_df = train.dtypes.reset_index()
dtype_df.columns = ["Count", "Column Type"] 
dtype_df.groupby("Column Type").aggregate('count').reset_index()

In [ ]:
dtype_df.loc[dtype_df["Column Type"] == "object",:] # -> "ID" and "COMPONENT_ARBITRARY" are the categorical columns.

In [ ]:
train.COMPONENT_ARBITRARY.value_counts().reset_index()

In [ ]:
plt.figure(figsize = (15,8))
sns.histplot(x = "Y_LABEL", data = train, hue = "COMPONENT_ARBITRARY", palette='CMRmap_r')
plt.title("Y_label", size=20, color = "red")

3) Missing values: Let us now check for the missing values.
- ?? : nan? ?? 
- nan? ?? ??? ?? test data set?? ??? ?? ?? ?? ?? 0?? ???. 

In [ ]:
#train.isnull().sum(axis=0).reset_index()
missing_df = train.isnull().sum(axis=0).reset_index()
missing_df.columns = ['column_name', 'missing_count']
mark = missing_df['missing_count']>0
missing_df = missing_df.loc[mark]
missing_df = missing_df.sort_values(by='missing_count')
missing_df

In [ ]:
train = train.fillna(0)
test = test.fillna(0)

4) Integer Columns Analysis

In [ ]:
# unique_values_dict = {}
# for col in train.columns:
#     if col not in ["ID", "Y_LABEL", "COMPONENT_ARBITRARY"]:
#         unique_value = str(np.sort(train[col].unique()).tolist())
#         tlist = unique_values_dict.get(unique_value, []) # dic.get(key, value) -> key? ??? ? ???? ?????.
#         tlist.append(col) # ?? key?? ?? ???? append??.  
#         unique_values_dict[unique_value] = tlist[:] # ???? ???? dic? ????. 
#         #print(unique_values_dict)
# for unique_val, columns in unique_values_dict.items():
#     print("Columns containing the unique values : ",unique_val)
#     print(columns)
#     print("--------------------------------------------------")

### 3. Data Preparation
- ?? ???? ???? ??? ????? : ?? ???? ? 4?? ?? ????.
- X_train & X_val data? groupby? ??? 4?? ????? ????.
- y_train & y_val data? ??? ??? target? ??. ??? ??? ??? ??? 4?? ????? ?????.

6) ??? ???? ???? ???? -> Random Forest ?? ? 



In [ ]:
from sklearn.model_selection import train_test_split
# X = train.drop(columns = "Y_LABEL")
# y = train.Y_LABEL
# X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# print('?? ? shape : ', X.shape)
# print('X_train? shape : ', X_train.shape)
# print('X_test? shape : ',X_test.shape)

print(len(train.loc[(train.COMPONENT_ARBITRARY == "COMPONENT1") & (train.Y_LABEL == 0)]))
print(len(train.loc[(train.COMPONENT_ARBITRARY == "COMPONENT1") & (train.Y_LABEL == 1)]))
print("-" * 5)
print(len(train.loc[(train.COMPONENT_ARBITRARY == "COMPONENT2") & (train.Y_LABEL == 0)]))
print(len(train.loc[(train.COMPONENT_ARBITRARY == "COMPONENT2") & (train.Y_LABEL == 1)]))
print("-" * 5)
print(len(train.loc[(train.COMPONENT_ARBITRARY == "COMPONENT3") & (train.Y_LABEL == 0)]))
print(len(train.loc[(train.COMPONENT_ARBITRARY == "COMPONENT3") & (train.Y_LABEL == 1)]))
print("-" * 5)
print(len(train.loc[(train.COMPONENT_ARBITRARY == "COMPONENT4") & (train.Y_LABEL == 0)]))
print(len(train.loc[(train.COMPONENT_ARBITRARY == "COMPONENT4") & (train.Y_LABEL == 1)]))

In [ ]:
train

In [ ]:
def grouping_df(target_df, categorical_feature) : 
    #print(df[categorical_feature])
    grouped_list = []
    for standard in list(target_df[categorical_feature].unique()) : 
        print(f"dividing my df on {standard}")
        grouped_list.append(target_df.loc[target_df.COMPONENT_ARBITRARY ==  standard, ].drop(columns = categorical_feature))
    return grouped_list

In [ ]:
test = grouping_df(train, "COMPONENT_ARBITRARY")

In [ ]:
test[0]

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from scipy.stats import skew, kurtosis

In [ ]:
print(len(train[train.Y_LABEL == 0])) # ??
print(len(train[train.Y_LABEL == 1])) # ???

In [ ]:
independent_variable = train.drop(columns = "Y_LABEL", )
dependent_variable = train.Y_LABEL
print(independent_variable.shape, dependent_variable.shape)

In [ ]:
# ????, ???? ??
independent_variable = train.drop(columns = ["Y_LABEL", "COMPONENT_ARBITRARY"])
dependent_variable = train.Y_LABEL
print(independent_variable.shape, dependent_variable.shape)

# summarize class distribution
print("before oversampling :")
#print(Counter(dependent_variable))
print("skewness", skew(dependent_variable)) 
print("kurtosis", kurtosis(dependent_variable, fisher=True)) 

# define oversampling strategy
oversample = RandomOverSampler(sampling_strategy='minority')
# fit and apply the transform
X_over, y_over = oversample.fit_resample(independent_variable, dependent_variable)

# summarize class distribution
print("\n" + "after oversampling :")
#print( Counter(y_over))
print("skewness", skew(y_over)) 
print("kurtosis", kurtosis(y_over, fisher=True)) 

In [ ]:
from collections import Counter
from sklearn.model_selection import train_test_split as tts
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

pca = PCA()
smt = SMOTE(random_state=42)
rfc = RFC()
pipeline = Pipeline([('smt', smt), ('pca', pca), ('rfc', rfc)])
X_train, X_test, y_train, y_test = tts(independent_variable, dependent_variable, random_state=42)
pipeline.fit(X_train, y_train) 
y_hat = pipeline.predict(X_test)
print(classification_report(y_test, y_hat))

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

steps = [('over', RandomOverSampler()), ('model', RandomForestClassifier())]
pipeline = Pipeline(steps=steps)
# evaluate pipeline
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
scores = cross_val_score(pipeline, X, y, scoring='f1_micro', cv=cv, n_jobs=-1)
score = mean(scores)
print('F1 Score: %.3f' % score)

In [ ]:
# ????
# validation dataset? test dataset? ??? ??? ??.
from sklearn.model_selection import train_test_split

def make_X_y(df) : 
    X = df.drop(columns = "Y_LABEL") # feature vector
    y = df.Y_LABEL # target value
    return X, y
  
def train_val(df, X_df, y_df) : 
  X, y = make_X_y(df)
  #print(X, y)
  X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
  return X_train, X_val, y_train, y_val

# tts ??? ???.
group_name = [group1, group2, group3, group4]
total_split_dic = {}
for num, name in enumerate(group_name) : 
  display(name)
  #print(make_X_y(name))
  X, y = make_X_y(name)
  X_train, X_val, y_train, y_val = train_val(df = name, X_df = X, y_df = y)
  total_split_dic[num] = X_train, X_val, y_train, y_val

total_split_dic.keys()

Random Forest Classifier

In [ ]:
# Random Forest ????
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

def random_forest_call(X_train, y_train, X_test, y_test):
  """
  main use will be supplement function for classifier_with_group
  """
  classifier_without_group = RandomForestClassifier()  # Random Forest ??? ????
  classifier_without_group.fit(X_train, y_train) # Random Forest ??? ?? parameter ???
  pred = classifier_without_group.predict(X_test) # Random Forest ???? ?? parameter ???
  acc = f1_score(y_test, pred)  # f1_score ??
  print("f1_score : %.3f" % acc)

  return classifier_without_group

def classifier_with_group(total_split_dic): 
  """
  return list that contains all classifier for each group 
  """
  classifier_list = []
  for key in total_split_dic : 
      X_train, X_test, y_train, y_test = total_split_dic[key]
      classifier_with_group = random_forest_call(X_train, y_train, X_test, y_test)
      classifier_list.append(classifier_with_group)

  return classifier_list

def check_feature_importance(classifier_list): 
    """
    return list of all feature_importance for each group 
    """
    feature_importance_list = []
    for classifier in classifier_list: 
        feature_importances_values = classifier.feature_importances_
        #print(len(X_train.columns))
        ftr_importances = pd.Series(feature_importances_values, index = X_train.columns).sort_values(ascending=False)
        feature_importance_list.append(ftr_importances)

    return feature_importance_list

# def draw_feature_importance(feature_importance_list) : 
#     """
#     return list of all feature_importance for each group 
#     """
#     for feature_importance in feature_importance_list : 
#         plt.figure(figsize=(12,6))
#         plt.title('Feature Importances')
#         sns.barplot(x = feature_importance, y = feature_importance.index)
#         plt.show()

def zero_importance_columns(feature_importance_list): 
    """
    return list of all feature_importance for each group 
    """
    zero_list = []
    for my_list in feature_importance_list: 
        df = my_list.reset_index()
        df.columns = ["name", "value"]
        zero_list.extend(df[df.value == 0].name.to_list())
        #print(zero_list, len(zero_list))

    return list(set(zero_list))

def drop_to_total(columns_list): 
    updated_group = []
    for group in group_name:
        group = group.drop(columns = intersection)
        updated_group.append(group)
  
    total_split_dic = {}
    for num, group in enumerate(updated_group): 
        X, y = make_X_y(group)
        X_train, X_val, y_train, y_val = train_val(df = group, X_df = X, y_df = y)
        total_split_dic[num] = X_train, X_val, y_train, y_val
        #total_split_dic.keys()

    return total_split_dic

In [ ]:
clf_with_group = classifier_with_group(classifier_with_group, total_split_dic)
print("clf_with_group")
print(clf_with_group)
print("-" * 100)

ftr_list = check_feature_importance(clf_with_group)
print("ftr_list")
print(ftr_list)
print("-" * 100)

draw_feature_importance(feature_importance_list = ftr_list) 
print("-" * 100)

zero_list = zero_importance_columns(feature_list = ftr_list)
print("zero_list")
print(zero_list, len(zero_list))
print("-" * 100)

In [ ]:
only_train_col = ["AL", "B", "BA", "BE", "CA", "CD", "FH2O", "FNOX", "FOPTIMETHGLY", "FOXID", "FSO4", "FTBN", "FUEL", "K", "LI", "MG", "NA", 
                  "P", "PB", "S", "SB", "SI", "SN", "SOOTPERCENTAGE", "U100", "U75", "U50", "U25", "U20", "U14", "U6", "U4", "V100"]
# print(len(only_train_col))
# print("-" * 100)

def check_column_relation(union = False, complement = False, intersection = True) : 
    if union == True : 
        union = list(set().union(only_train_col,zero_list))
        return union

    if complement == True : 
        complement = list(set(only_train_col).difference(zero_list))
        return complement

    if intersection == True : 
        intersection = list(set(only_train_col).intersection(zero_list))
        return intersection

In [ ]:
intersection = check_column_relation(union = False, complement = False, intersection = True)

??? 0? ??? ???? ?? ???

In [ ]:
updated_group = []
for group in group_name :
    group = group.drop(columns = intersection)
    updated_group.append(group)

updated_group

#group_name = [group1, group2, group3, group4]
total_split_dic = {}
for num, group in enumerate(updated_group) : 
    X, y = make_X_y(group)
    X_train, X_val, y_train, y_val = train_val(df = group, X_df = X, y_df = y)
    total_split_dic[num] = X_train, X_val, y_train, y_val

total_split_dic.keys()

In [ ]:
clf_with_group = classifier_with_group(classifier_with_group, total_split_dic)
print("clf_with_group")
print(clf_with_group)
print("-" * 100)

ftr_list = check_feature_importance(clf_with_group)
print("ftr_list")
print(ftr_list)
print("-" * 100)

draw_feature_importance(feature_importance_list = ftr_list) 
print("-" * 100)

zero_list = zero_importance_columns(feature_list = ftr_list)
print("zero_list")
print(zero_list, len(zero_list))
print("-" * 100)

In [ ]:
zero_list

??? ?? -> ?? ??

In [ ]:
# updated_group = []
# for group in group_name :
#     #print(num)
#     #display(each.drop(columns = intersection))
#     group = group.drop(columns = zero_list)
#     updated_group.append(group)

# print(updated_group)
# # total_split_dic = {}
# # for num, group in enumerate(updated_group) : 
# #     X, y = make_X_y(group)
# #     X_train, X_val, y_train, y_val = train_val(df = group, X_df = X, y_df = y)
# #     total_split_dic[num] = X_train, X_val, y_train, y_val

# print(updated_group[0].columns)

# clf_with_group = classifier_with_group(classifier_with_group, total_split_dic)
# print("clf_with_group")
# print(clf_with_group)
# print("-" * 100)

# ftr_list = check_feature_importance(clf_with_group)
# print("ftr_list")
# print(ftr_list)
# print("-" * 100)

# draw_feature_importance(feature_importance_list = ftr_list) 
# print("-" * 100)

# zero_list = zero_importance_columns(feature_list = ftr_list)
# print("zero_list")
# print(zero_list, len(zero_list))
# print("-" * 100)